# 04｜ViT Retrain 與測試

載入 `best_vit_model.pth`，使用 `dataset_vit_retrain/train` 與
`dataset_vit_retrain/val` 繼續訓練，並在最後載入最佳 Retrain 權重，
對 `dataset_vit_retrain/test` 評估。

Test 標籤直接取自 `test/fake`、`test/real` 資料夾，檔名不限；
測試圖片已由第三份 Notebook 裁臉，因此這裡不會重複執行 YOLO。


In [ ]:
# =========================
# 參數設定區
# =========================
DATASET_DIR = "dataset_vit_retrain"
MODEL_NAME = "vit_base_patch16_224"
BASE_MODEL_PATH = "best_vit_model.pth"
BEST_RETRAIN_MODEL_PATH = "best_vit_model_retrain.pth"
IMAGE_SIZE = 224
NUM_CLASSES = 2

EPOCHS = 12
BATCH_SIZE = 8
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
RANDOM_SEED = 42
USE_AMP = True

# Retrain 訓練資料增強（依原本實際啟用的設定）
RANDOM_CROP_SCALE = (0.8, 1.0)
HORIZONTAL_FLIP_PROB = 0.5

USE_COLOR_JITTER = True
COLOR_JITTER_PROB = 0.6
COLOR_JITTER_BRIGHTNESS = 0.25
COLOR_JITTER_CONTRAST = 0.25
COLOR_JITTER_SATURATION = 0.10
COLOR_JITTER_HUE = 0.01

USE_GAUSSIAN_BLUR = True
GAUSSIAN_BLUR_PROB = 0.05
GAUSSIAN_BLUR_KERNEL_SIZE = 3
GAUSSIAN_BLUR_SIGMA = (0.1, 0.6)

# 隨機 JPEG 壓縮：模擬社群平台上傳後的壓縮痕跡。
# 每張圖片最多套用一次，不重複壓縮。
USE_JPEG_COMPRESSION = True
JPEG_COMPRESSION_PROB = 0.4
JPEG_QUALITY_MIN = 60
JPEG_QUALITY_MAX = 90

USE_WHITE_BALANCE = True
WHITE_BALANCE_PROB = 0.8

# 0 表示整個模型都參與 Retrain；
# 大於 0 表示只訓練最後 N 個 Transformer blocks 與分類頭
NUM_UNFROZEN_BLOCKS = 0

HISTORY_CSV = "retrain_history.csv"
HISTORY_FIGURE = "retrain_result.png"
TEST_RESULTS_CSV = "retrain_test_results.csv"
CONFUSION_MATRIX_FIGURE = "confusion_matrix_retrain_test.png"


In [ ]:
import io
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
)
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm
import timm

import matplotlib.pyplot as plt

# 中文圖表字型：依目前電腦已安裝字型自動擇一
plt.rcParams["font.sans-serif"] = [
    "Microsoft JhengHei", "Noto Sans CJK TC", "PingFang TC",
    "Arial Unicode MS", "DejaVu Sans"
]
plt.rcParams["axes.unicode_minus"] = False



def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class RandomJPEGCompression:
    def __init__(self, probability, quality_min, quality_max):
        self.probability = probability
        self.quality_min = quality_min
        self.quality_max = quality_max

    def __call__(self, image):
        if random.random() >= self.probability:
            return image
        quality = random.randint(self.quality_min, self.quality_max)
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=quality)
        buffer.seek(0)
        return Image.open(buffer).convert("RGB").copy()


class RandomWhiteBalance:
    def __init__(self, probability):
        self.probability = probability

    def __call__(self, image_tensor):
        if torch.rand(1).item() >= self.probability:
            return image_tensor
        channel_means = image_tensor.mean(dim=(1, 2), keepdim=True)
        gray_mean = channel_means.mean()
        scale = gray_mean / channel_means.clamp(min=1e-6)
        return torch.clamp(image_tensor * scale, 0.0, 1.0)


def create_transforms(image_size):
    mean = [0.485, 0.456, 0.406]
    std = [0.229, 0.224, 0.225]

    train_steps = [
        transforms.RandomResizedCrop(image_size, scale=RANDOM_CROP_SCALE),
        transforms.RandomHorizontalFlip(p=HORIZONTAL_FLIP_PROB),
    ]

    if USE_COLOR_JITTER:
        train_steps.append(transforms.RandomApply([
            transforms.ColorJitter(
                brightness=COLOR_JITTER_BRIGHTNESS,
                contrast=COLOR_JITTER_CONTRAST,
                saturation=COLOR_JITTER_SATURATION,
                hue=COLOR_JITTER_HUE,
            )
        ], p=COLOR_JITTER_PROB))

    if USE_GAUSSIAN_BLUR:
        train_steps.append(transforms.RandomApply([
            transforms.GaussianBlur(
                kernel_size=GAUSSIAN_BLUR_KERNEL_SIZE,
                sigma=GAUSSIAN_BLUR_SIGMA,
            )
        ], p=GAUSSIAN_BLUR_PROB))

    if USE_JPEG_COMPRESSION:
        train_steps.append(RandomJPEGCompression(
            probability=JPEG_COMPRESSION_PROB,
            quality_min=JPEG_QUALITY_MIN,
            quality_max=JPEG_QUALITY_MAX,
        ))

    train_steps.append(transforms.ToTensor())

    if USE_WHITE_BALANCE:
        train_steps.append(RandomWhiteBalance(WHITE_BALANCE_PROB))

    train_steps.append(transforms.Normalize(mean, std))
    train_transform = transforms.Compose(train_steps)

    eval_transform = transforms.Compose([
        transforms.Resize(image_size + 32),
        transforms.CenterCrop(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    return train_transform, eval_transform


def create_loaders(data_dir, image_size, batch_size, workers):
    train_tf, eval_tf = create_transforms(image_size)
    datasets_by_split = {
        "train": datasets.ImageFolder(Path(data_dir) / "train", train_tf),
        "val": datasets.ImageFolder(Path(data_dir) / "val", eval_tf),
        "test": datasets.ImageFolder(Path(data_dir) / "test", eval_tf),
    }
    expected = datasets_by_split["train"].class_to_idx
    for split, dataset in datasets_by_split.items():
        if dataset.class_to_idx != expected:
            raise ValueError(f"{split} 類別順序不一致：{dataset.class_to_idx} vs {expected}")
    loaders = {
        split: DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=(split == "train"),
            num_workers=workers,
            pin_memory=torch.cuda.is_available(),
        )
        for split, dataset in datasets_by_split.items()
    }
    print("類別對應：", expected)
    print("資料數量：", {k: len(v) for k, v in datasets_by_split.items()})
    return loaders, datasets_by_split["train"].classes


def run_epoch(model, loader, criterion, device, optimizer=None, use_amp=True):
    training = optimizer is not None
    model.train(training)
    amp_enabled = use_amp and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    losses, labels_all, predictions_all = 0.0, [], []

    for images, labels in tqdm(loader, leave=False):
        images, labels = images.to(device), labels.to(device)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=amp_enabled):
            logits = model(images)
            loss = criterion(logits, labels)
        if training:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        losses += loss.item() * images.size(0)
        labels_all.extend(labels.detach().cpu().tolist())
        predictions_all.extend(logits.argmax(dim=1).detach().cpu().tolist())

    return {
        "loss": losses / len(loader.dataset),
        "accuracy": accuracy_score(labels_all, predictions_all),
        "f1": f1_score(labels_all, predictions_all, average="macro", zero_division=0),
    }


def train_model(model, loaders, device, epochs, learning_rate, weight_decay,
                save_path, use_amp):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=learning_rate,
        weight_decay=weight_decay,
    )
    best_accuracy = -1.0
    history = []

    for epoch in range(1, epochs + 1):
        start = time.time()
        train_metrics = run_epoch(
            model, loaders["train"], criterion, device, optimizer, use_amp
        )
        with torch.no_grad():
            val_metrics = run_epoch(
                model, loaders["val"], criterion, device, None, use_amp
            )
        row = {"epoch": epoch}
        row.update({f"train_{k}": v for k, v in train_metrics.items()})
        row.update({f"val_{k}": v for k, v in val_metrics.items()})
        history.append(row)
        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"train loss={train_metrics['loss']:.4f}, acc={train_metrics['accuracy']:.4f}, "
            f"F1={train_metrics['f1']:.4f} | "
            f"val loss={val_metrics['loss']:.4f}, acc={val_metrics['accuracy']:.4f}, "
            f"F1={val_metrics['f1']:.4f} | {time.time() - start:.1f} 秒"
        )
        if val_metrics["accuracy"] > best_accuracy:
            best_accuracy = val_metrics["accuracy"]
            torch.save(model.state_dict(), save_path)
            print(f"已更新最佳模型：{save_path}")
    return pd.DataFrame(history)


def plot_history(history, save_path):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, metric, title in zip(
        axes, ("loss", "accuracy", "f1"), ("Loss", "Accuracy", "Macro F1")
    ):
        ax.plot(history["epoch"], history[f"train_{metric}"], label="Train")
        ax.plot(history["epoch"], history[f"val_{metric}"], label="Validation")
        ax.set_title(title)
        ax.set_xlabel("Epoch")
        ax.grid(alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


def load_state(path, device):
    try:
        return torch.load(path, map_location=device, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=device)


def evaluate_model(model, loader, class_names, device, use_amp, csv_path, cm_path):
    model.eval()
    amp_enabled = use_amp and device.type == "cuda"
    rows, y_true, y_pred = [], [], []
    paths = [path for path, _ in loader.dataset.samples]
    offset = 0
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Test"):
            images = images.to(device)
            with torch.amp.autocast("cuda", enabled=amp_enabled):
                probabilities = torch.softmax(model(images), dim=1).cpu()
            predictions = probabilities.argmax(dim=1)
            for i in range(len(labels)):
                true_idx = int(labels[i])
                pred_idx = int(predictions[i])
                row = {
                    "file": paths[offset + i],
                    "true_label": class_names[true_idx],
                    "pred_label": class_names[pred_idx],
                    "confidence": float(probabilities[i, pred_idx]),
                }
                for class_index, class_name in enumerate(class_names):
                    row[f"p_{class_name}"] = float(probabilities[i, class_index])
                rows.append(row)
            offset += len(labels)
            y_true.extend(labels.tolist())
            y_pred.extend(predictions.tolist())

    pd.DataFrame(rows).to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"Recall   : {recall_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print(f"Macro F1 : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
    print("\nClassification Report\n")
    print(classification_report(
        y_true, y_pred, labels=range(len(class_names)),
        target_names=class_names, digits=4, zero_division=0
    ))

    matrix = confusion_matrix(y_true, y_pred, labels=range(len(class_names)))
    fig, ax = plt.subplots(figsize=(6, 5))
    image = ax.imshow(matrix, cmap="Blues")
    fig.colorbar(image, ax=ax)
    ax.set(
        xticks=range(len(class_names)), yticks=range(len(class_names)),
        xticklabels=class_names, yticklabels=class_names,
        xlabel="預測類別", ylabel="真實類別", title="混淆矩陣",
    )
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(j, i, matrix[i, j], ha="center", va="center")
    plt.tight_layout()
    plt.savefig(cm_path, dpi=200, bbox_inches="tight")
    plt.show()
    return pd.DataFrame(rows)



def configure_trainable_layers(model, num_unfrozen_blocks):
    if num_unfrozen_blocks == 0:
        for parameter in model.parameters():
            parameter.requires_grad = True
        return
    if not hasattr(model, "blocks"):
        raise AttributeError("這個模型沒有 blocks，無法依 Transformer block 解凍")
    if not 1 <= num_unfrozen_blocks <= len(model.blocks):
        raise ValueError(
            f"NUM_UNFROZEN_BLOCKS 必須介於 0 到 {len(model.blocks)}"
        )
    for parameter in model.parameters():
        parameter.requires_grad = False
    for block in model.blocks[-num_unfrozen_blocks:]:
        for parameter in block.parameters():
            parameter.requires_grad = True
    for name in ("norm", "fc_norm", "head"):
        module = getattr(model, name, None)
        if module is not None:
            for parameter in module.parameters():
                parameter.requires_grad = True


In [ ]:
# =========================
# 建立資料並載入初始權重
# =========================
set_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("使用裝置：", device)

loaders, class_names = create_loaders(
    DATASET_DIR, IMAGE_SIZE, BATCH_SIZE, NUM_WORKERS
)
if len(class_names) != NUM_CLASSES:
    raise ValueError(f"資料集有 {len(class_names)} 類，但 NUM_CLASSES={NUM_CLASSES}")

model = timm.create_model(
    MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES
).to(device)
model.load_state_dict(load_state(BASE_MODEL_PATH, device))
configure_trainable_layers(model, NUM_UNFROZEN_BLOCKS)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"可訓練參數：{trainable:,} / {total:,}")


In [ ]:
# =========================
# 開始 Retrain
# =========================
history = train_model(
    model=model,
    loaders=loaders,
    device=device,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    save_path=BEST_RETRAIN_MODEL_PATH,
    use_amp=USE_AMP,
)
history.to_csv(HISTORY_CSV, index=False, encoding="utf-8-sig")
plot_history(history, HISTORY_FIGURE)
history.tail()


In [ ]:
# =========================
# 載入最佳 Retrain 模型並測試
# =========================
best_model = timm.create_model(
    MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES
).to(device)
best_model.load_state_dict(load_state(BEST_RETRAIN_MODEL_PATH, device))

test_results = evaluate_model(
    best_model, loaders["test"], class_names, device, USE_AMP,
    TEST_RESULTS_CSV, CONFUSION_MATRIX_FIGURE
)
test_results.head(10)
